# STG_6 — Comparación de modelos de predicción de voto

Toma `data/df_entrenamiento.csv` y compara cinco enfoques de clasificación
usando validación cruzada temporal (`TimeSeriesSplit`). Métrica principal: **F1-macro**.

Modelos comparados (de simple a complejo):
1. DummyClassifier (baseline trivial)
2. LogisticRegression (baseline interpretable)
3. RandomForestClassifier
4. BaggingClassifier
5. XGBClassifier
6. LGBMClassifier


In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

DATA_DIR    = Path("../data")
SPEC_DIR    = Path("../specs/009-modelado-prediccion-votos")
RANDOM_STATE = 42


## T12 — Carga y preparación

In [ ]:
# T12 — Cargar dataset, definir features y target
df = pd.read_csv(DATA_DIR / "df_entrenamiento.csv", parse_dates=["fecha_votacion"])
df = df.sort_values("fecha_votacion").reset_index(drop=True)

# Features: todo excepto columnas meta y el target
META   = ["diputado", "titulo_base", "fecha_votacion", "bloque", "provincia", "tema_label"]
TARGET = "voto"
FEATURES = [c for c in df.columns if c not in META + [TARGET]]

print(f"Filas    : {len(df):,}")
print(f"Features : {len(FEATURES)}")
print(f"Target   : {TARGET}")
print()

# Codificar el target con LabelEncoder y guardarlo
from sklearn.preprocessing import LabelEncoder
le_voto = LabelEncoder()
df["voto_enc"] = le_voto.fit_transform(df[TARGET])
joblib.dump(le_voto, DATA_DIR / "le_voto.joblib")

print("Clases del target:")
for i, cls in enumerate(le_voto.classes_):
    n = (df["voto_enc"] == i).sum()
    print(f"  {i} = {cls}: {n:,} ({n/len(df)*100:.1f}%)")

print()
assert df[FEATURES].isna().sum().sum() == 0, "Hay NaN en features"
assert len(df) == 25082
print("T12 PASA.")


## T13 — Split temporal y TimeSeriesSplit

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Particion pasado/futuro: el 80% mas antiguo entrena, el 20% mas reciente es el holdout
corte_idx = int(len(df) * 0.80)
fecha_corte = df.iloc[corte_idx]["fecha_votacion"]

df_train = df.iloc[:corte_idx].copy()
df_hold  = df.iloc[corte_idx:].copy()

X_train = df_train[FEATURES].values
y_train = df_train["voto_enc"].values
X_hold  = df_hold[FEATURES].values
y_hold  = df_hold["voto_enc"].values

tscv = TimeSeriesSplit(n_splits=5)

print(f"Fecha de corte train/holdout : {fecha_corte.date()}")
print(f"Train : {len(df_train):,} filas ({df_train['fecha_votacion'].min().date()} a {df_train['fecha_votacion'].max().date()})")
print(f"Holdout: {len(df_hold):,} filas ({df_hold['fecha_votacion'].min().date()} a {df_hold['fecha_votacion'].max().date()})")
print()
print("Distribucion de clases en holdout:")
for i, cls in enumerate(le_voto.classes_):
    n = (y_hold == i).sum()
    print(f"  {cls}: {n:,} ({n/len(y_hold)*100:.1f}%)")
print()

# Verificacion: ninguna fecha del train supera la minima del holdout
assert df_train["fecha_votacion"].max() <= df_hold["fecha_votacion"].min(),     "VIOLATION: hay fechas del train posteriores al holdout"

# Verificar que TimeSeriesSplit respeta orden: cada fold valida con datos mas nuevos que el train
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    max_tr  = df_train.iloc[tr_idx]["fecha_votacion"].max()
    min_val = df_train.iloc[val_idx]["fecha_votacion"].min()
    assert max_tr <= min_val, f"Fold {fold}: fechas del train posteriores a la validacion"

print(f"TimeSeriesSplit con {tscv.n_splits} folds verificado: orden temporal correcto en todos los folds.")
print("T13 PASA.")


## T14 — Baseline: DummyClassifier y LogisticRegression

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

resultados = {}  # acumula resultados de todos los modelos

def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_ho, y_ho, tscv, cv_n_jobs=-1):
    """Evalua un modelo con CV temporal y en el holdout. Devuelve dict con metricas."""
    scores_cv = cross_val_score(modelo, X_tr, y_tr, cv=tscv, scoring='f1_macro', n_jobs=cv_n_jobs)
    modelo.fit(X_tr, y_tr)
    y_pred_hold = modelo.predict(X_ho)
    f1_hold = f1_score(y_ho, y_pred_hold, average='macro')
    print(f"  {nombre:<30} CV F1-macro: {scores_cv.mean():.3f} ± {scores_cv.std():.3f} | Holdout: {f1_hold:.3f}")
    return {'modelo': nombre, 'cv_mean': scores_cv.mean(), 'cv_std': scores_cv.std(), 'holdout': f1_hold, 'estimator': modelo}

print("Entrenando modelos baseline...")
print()

# 1. DummyClassifier — predice siempre la clase mas frecuente
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
resultados['Dummy'] = evaluar_modelo('Dummy (most_frequent)', dummy, X_train, y_train, X_hold, y_hold, tscv)

# 2. LogisticRegression — requiere escalar features (los embeddings tienen distinta magnitud)
logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
])
resultados['LogReg'] = evaluar_modelo('LogisticRegression', logreg_pipe, X_train, y_train, X_hold, y_hold, tscv)

print()
print("T14 PASA.")


## T15 — RandomForestClassifier y BaggingClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

print("Entrenando modelos de ensemble (puede tardar unos minutos)...")
print()

# RandomForest: muchos arboles de decision entrenados sobre subsets aleatorios de features
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
resultados['RF'] = evaluar_modelo('RandomForest', rf, X_train, y_train, X_hold, y_hold, tscv)

# Bagging: cada arbol ve un subset aleatorio de FILAS (no de features)
bag = BaggingClassifier(estimator=DecisionTreeClassifier(class_weight='balanced'),
                        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
resultados['Bagging'] = evaluar_modelo('BaggingClassifier', bag, X_train, y_train, X_hold, y_hold, tscv)

print()
print("T15 PASA.")


from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("Entrenando modelos de boosting (puede tardar unos minutos)...")
print()

xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
)
resultados['XGB'] = evaluar_modelo('XGBClassifier', xgb, X_train, y_train, X_hold, y_hold, tscv)

# LightGBM: cv_n_jobs=1 para evitar doble paralelismo (LightGBM ya paraleliza internamente)
lgbm = LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)
resultados['LGBM'] = evaluar_modelo('LGBMClassifier', lgbm, X_train, y_train, X_hold, y_hold, tscv,
                                     cv_n_jobs=1)

print()
print("T16 PASA.")


In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("Entrenando modelos de boosting (puede tardar unos minutos)...")
print()

# XGBoost: boosting con regularizacion L1/L2; maneja bien muchas features
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)
resultados['XGB'] = evaluar_modelo('XGBClassifier', xgb, X_train, y_train, X_hold, y_hold, tscv)

# LightGBM: boosting por hojas (leaf-wise), mas rapido y eficiente con muchas features
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)
resultados['LGBM'] = evaluar_modelo('LGBMClassifier', lgbm, X_train, y_train, X_hold, y_hold, tscv)

print()
print("T16 PASA.")


## T17 — Tabla comparativa y elección del ganador

In [ ]:
import pandas as pd

# Tabla comparativa
tabla = pd.DataFrame([
    {
        'Modelo': v['modelo'],
        'CV F1-macro (media)': round(v['cv_mean'], 3),
        'CV F1-macro (desvio)': round(v['cv_std'], 3),
        'Holdout F1-macro': round(v['holdout'], 3),
    }
    for v in resultados.values()
]).sort_values('Holdout F1-macro', ascending=False).reset_index(drop=True)

print("=" * 65)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 65)
print(tabla.to_string(index=False))
print("=" * 65)

# Guardar tabla
tabla.to_csv(SPEC_DIR / "tabla_comparativa_modelos.csv", index=False)
print()
print(f"Tabla guardada en specs/009-modelado-prediccion-votos/tabla_comparativa_modelos.csv")

# Elegir ganador por F1-macro en holdout
nombre_ganador = tabla.iloc[0]['Modelo']
ganador = next(v['estimator'] for v in resultados.values() if v['modelo'] == nombre_ganador)

print()
print(f"GANADOR: {nombre_ganador}  (Holdout F1-macro = {tabla.iloc[0]['Holdout F1-macro']})")
print("T17 PASA.")


## T18 — Gráficos: comparación de modelos y matriz de confusión

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# --- Gráfico 1: comparación de modelos (barras horizontales) ---
tabla_plot = tabla.sort_values('Holdout F1-macro')
colores_cv   = ['#5B9BD5'] * len(tabla_plot)
colores_hold = ['#ED7D31'] * len(tabla_plot)

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(tabla_plot))
h = 0.35
ax.barh(y + h/2, tabla_plot['CV F1-macro (media)'],   h, color='#5B9BD5', label='CV F1-macro (media)')
ax.barh(y - h/2, tabla_plot['Holdout F1-macro'],       h, color='#ED7D31', label='Holdout F1-macro')
ax.set_yticks(y)
ax.set_yticklabels(tabla_plot['Modelo'], fontsize=10)
ax.set_xlabel('F1-macro')
ax.set_title('Comparación de modelos — F1-macro', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.legend(loc='lower right')
ax.set_xlim(0, 0.6)
for bar in ax.patches:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height()/2,
            f'{w:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(SPEC_DIR / 'comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.close()
print("Guardado: comparacion_modelos.png")

# --- Gráfico 2: matriz de confusión del ganador en holdout ---
y_pred_hold = ganador.predict(X_hold)
cm = confusion_matrix(y_hold, y_pred_hold)
clases = le_voto.classes_

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clases)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de confusión — {nombre_ganador}\n'
            f'(holdout: {df_hold["fecha_votacion"].min().date()} a {df_hold["fecha_votacion"].max().date()})',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(SPEC_DIR / 'matriz_confusion_ganador.png', dpi=150, bbox_inches='tight')
plt.close()
print("Guardado: matriz_confusion_ganador.png")

# Imprimir resumen de la matriz
print()
print("Matriz de confusion (filas=real, columnas=predicho):")
print(f"  Clases: {list(clases)}")
for i, fila in enumerate(cm):
    print(f"  {clases[i]:<12}: {list(fila)}")

print()
print("T18 PASA.")


## T19 — Importancia de features del modelo ganador

In [ ]:
# LightGBM expone feature_importances_ en dos modos:
# 'split': cuantas veces se uso la feature para dividir un nodo
# 'gain' : cuanto ganó en promedio cada split (mas representativo de utilidad real)
importancias = pd.DataFrame({
    'feature': FEATURES,
    'importancia_gain': ganador.feature_importances_  # LightGBM usa 'gain' por defecto
}).sort_values('importancia_gain', ascending=False).reset_index(drop=True)

# Normalizar a porcentaje
importancias['importancia_pct'] = (
    importancias['importancia_gain'] / importancias['importancia_gain'].sum() * 100
)

# Guardar tabla completa
importancias.to_csv(SPEC_DIR / 'importancia_features.csv', index=False)
print("Guardado: importancia_features.csv")

# --- Resumen por grupo ---
emb_cols_set = set(f for f in FEATURES if f.startswith('emb_'))
hist_cols = ['tasa_afirmativo_hist','tasa_afirmativo_tema_hist','tasa_alineacion_bloque_hist',
             'tasa_afirmativo_desde_2023','tasa_afirmativo_2026','n_votos_hist',
             'tasa_afirmativo_bloque_tema']
enc_cols  = ['bloque_enc','provincia_enc']

def pct_grupo(cols):
    return importancias[importancias['feature'].isin(cols)]['importancia_pct'].sum()

print()
print("Importancia por grupo:")
print(f"  Historicas del diputado/bloque : {pct_grupo(hist_cols + enc_cols):.1f}%")
print(f"  Embeddings semanticos (384 dim): {pct_grupo(emb_cols_set):.1f}%")
print(f"  tema_id                        : {pct_grupo(['tema_id']):.1f}%")
print()
print("Top 15 features individuales:")
print(importancias.head(15)[['feature','importancia_pct']].to_string(index=False))

# --- Grafico 1: Top 30 features ---
top30 = importancias.head(30).sort_values('importancia_pct')
colores = ['#ED7D31' if f in set(hist_cols + enc_cols + ['tema_id'])
           else '#5B9BD5' for f in top30['feature']]

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(top30['feature'], top30['importancia_pct'], color=colores)
ax.set_xlabel('Importancia (%)')
ax.set_title('Top 30 features por importancia — LGBMClassifier', fontsize=12, fontweight='bold')
from matplotlib.patches import Patch
leyenda = [Patch(color='#ED7D31', label='Features historicas / encoding / tema'),
           Patch(color='#5B9BD5', label='Embeddings semanticos')]
ax.legend(handles=leyenda, loc='lower right')
for bar, val in zip(bars, top30['importancia_pct']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, f'{val:.2f}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(SPEC_DIR / 'importancia_features_top30.png', dpi=150, bbox_inches='tight')
plt.close()
print()
print("Guardado: importancia_features_top30.png")

# --- Grafico 2: comparacion grupos (torta / barras) ---
grupos = {
    'Historicas\ndiputado/bloque': pct_grupo(hist_cols + enc_cols),
    'Embeddings\n(384 dim)': pct_grupo(emb_cols_set),
    'tema_id': pct_grupo(['tema_id']),
}
fig, ax = plt.subplots(figsize=(7, 4))
colores_g = ['#ED7D31','#5B9BD5','#70AD47']
bars_g = ax.bar(grupos.keys(), grupos.values(), color=colores_g, width=0.5)
ax.set_ylabel('Importancia (%)')
ax.set_title('Importancia por grupo de features — LGBMClassifier', fontsize=12, fontweight='bold')
for bar, val in zip(bars_g, grupos.values()):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_ylim(0, max(grupos.values()) * 1.15)
plt.tight_layout()
plt.savefig(SPEC_DIR / 'importancia_por_grupo.png', dpi=150, bbox_inches='tight')
plt.close()
print("Guardado: importancia_por_grupo.png")
print()
print("T19 PASA.")
